# AEGIS-SQL — Spider-KO mschema + 1회 bounded repair

Qwen2.5-Coder-1.5B-Instruct + NF4 4-bit + `mschema`를 고정하고, 선택된 SQLite 실행 오류에만 **최대 1회** repair를 허용합니다.

> 결과는 **공식 Spider leaderboard** 점수가 아니라 AEGIS `execution_match` 기반 외부 일반화 실험입니다. repair 입력에는 gold SQL이 들어가지 않습니다.


In [ ]:
import subprocess
import torch

subprocess.run(["nvidia-smi"], check=True)
if not torch.cuda.is_available():
    raise RuntimeError("GPU 런타임이 필요합니다. Colab 런타임 설정에서 T4/L4 GPU를 선택하세요.")
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
ARCHIVE_DIR = Path("/content/drive/MyDrive/AEGIS-SQL")
RUN_DIR = ARCHIVE_DIR / "spider-ko-bounded-repair"
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)
print("persistent archive:", RUN_DIR)


In [ ]:
import shutil
import subprocess
from pathlib import Path

REPO_DIR = Path("/content/aegis-sql")
REPO_URL = "https://github.com/sokldjs554/aegis-sql.git"
REF = "main"
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", "--depth", "1", "--branch", REF, REPO_URL, str(REPO_DIR)], check=True)
print("repo SHA:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip())


In [ ]:
import os
import subprocess

env = os.environ.copy()
env["ARCHIVE_DIR"] = str(ARCHIVE_DIR)
subprocess.run(
    ["bash", "scripts/run_spider_ko_bounded_repair_colab.sh"],
    cwd=REPO_DIR,
    env=env,
    check=True,
)


In [ ]:
import json

REPORT = RUN_DIR / "spider-ko-mschema-bounded-repair.json"
report = json.loads(REPORT.read_text(encoding="utf-8"))
initial = report["initial_execution_accuracy"]
final = report["execution_accuracy"]
repair = report["repair"]
print(f"initial EX: {report['initial_correct']}/{report['items']} = {initial:.2%}")
print(f"final EX:   {report['correct']}/{report['items']} = {final:.2%}")
print(f"delta:      {(final - initial) * 100:+.2f} pp")
print("initial/final execution failures:", report["initial_execution_failures"], "->", report["execution_failures"])
print("repair:", json.dumps(repair, ensure_ascii=False, indent=2))
print("report:", REPORT)
